In [1]:
import pandas as pd
from pathlib import Path

# Fil
DATA = Path("data/raw/fra-excel.txt")

# Læs tekstfil
df = pd.read_csv(DATA, sep="\t")

# Kolonnen hedder "Adresse"
df["Adresse"] = df["Adresse"].str.strip()

# Split "HM 11, kld. tv."
tmp = df["Adresse"].str.extract(
    r"HM\s+(?P<opgang>\d+),\s*(?P<etage>kld\.|st\.|\d+\.)\s*(?P<side>tv\.|th\.)"
)

df = pd.concat([df, tmp], axis=1)

# Fjern punktummer
df["etage"] = df["etage"].str.replace(".", "", regex=False)
df["side"] = df["side"].str.replace(".", "", regex=False)

# Visning
df["visning"] = (
    df["opgang"]
    + ", "
    + df["etage"]
    + ". "
    + df["side"]
    + "."
)

# Mappenavn
etage_map = {
    "kld": "_kld",
    "st": "_st",
}

df["mappenavn"] = (
    df["opgang"]
    + "-"
    + df["side"]
    + "-"
    + df["etage"].replace(etage_map)
)

df["har_lokaler"] = ~df["etage"].eq("kld")

print(df[["opgang", "etage", "side", "visning", "mappenavn", "har_lokaler"]])

   opgang etage side       visning   mappenavn  har_lokaler
0      11   kld   tv  11, kld. tv.  11-tv-_kld        False
1      11   kld   th  11, kld. th.  11-th-_kld        False
2      11    st   tv   11, st. tv.   11-tv-_st         True
3      11    st   th   11, st. th.   11-th-_st         True
4      11     1   tv    11, 1. tv.     11-tv-1         True
..    ...   ...  ...           ...         ...          ...
57     29    st   th   29, st. th.   29-th-_st         True
58     29     1   tv    29, 1. tv.     29-tv-1         True
59     29     1   th    29, 1. th.     29-th-1         True
60     29     2   tv    29, 2. tv.     29-tv-2         True
61     29     2   th    29, 2. th.     29-th-2         True

[62 rows x 6 columns]


In [2]:
from pathlib import Path

# Outputmappe
OUTPUT_DIR = Path("/workspace/data/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Filnavn
output_file = OUTPUT_DIR / "mappenavne.txt"

# Gem én linje pr. mappenavn
df["mappenavn"].to_csv(
    output_file,
    index=False,
    header=False
)

print(f"Skrev {len(df)} mappenavne til")
print(output_file)

Skrev 62 mappenavne til
/workspace/data/output/mappenavne.txt


In [7]:
# spring over

from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas

pdf_fil = "data/output/lejligheder-test.pdf"

c = canvas.Canvas(pdf_fil, pagesize=A4)
bredde, højde = A4

for _, row in df.iterrows():
    c.setFont("Helvetica-Bold", 24)
    c.drawCentredString(bredde / 2, højde / 2, row["visning"])
    c.showPage()

c.save()

print(f"PDF gemt som {pdf_fil}")

PDF gemt som data/output/lejligheder.pdf


In [3]:
# with open("data/raw/ventiler.txt", encoding="utf-8") as f:
#    ventiler = [linje.strip() for linje in f if linje.strip()]
#
# ventiler

In [4]:
df_ventiler = pd.read_csv(
    "data/raw/ventiler.txt",
    sep = "\t",
    header=None,
    names=["lokale"]
)

df_ventiler

,lokale
0,Køkken
1,Soveværelse
2,Badeværelse
3,Stue
4,"Nordvendt værelse, nærmest"
5,"Nordvendt værelse, fjernest"


In [11]:
# SPRING OVER

from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas

pdf_fil = "data/output/lejligheder.pdf"

c = canvas.Canvas(pdf_fil, pagesize=A4)
bredde, højde = A4

tekst = (
    "Lorem ipsum dolor sit amet, consectetur adipiscing elit. "
    "Sed do eiusmod tempor incididunt ut labore et dolore magna aliqua."
)

for _, row in df.iterrows():

    # Lejlighedens navn
    c.setFont("Helvetica-Bold", 20)
    c.drawString(40, højde - 40, row["visning"])

    # Instruktion
    text = c.beginText()
    text.setTextOrigin(40, højde - 70)
    text.setFont("Helvetica", 11)

    for linje in tekst.split(". "):
        text.textLine(linje)

    c.drawText(text)

    # Streg under instruktionerne
    c.line(40, højde - 130, bredde - 40, højde - 130)

    c.showPage()

c.save()

In [5]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm
from reportlab.pdfgen import canvas
import textwrap

# ---------- PDF ----------
pdf_fil = "data/output/Ventilsedler.pdf"

c = canvas.Canvas(pdf_fil, pagesize=A4)

c.setTitle("Ventilsedler")
c.setAuthor("A/B Møllevænget")
c.setSubject("Fotografering af radiatorventiler")
c.setCreator("Python / ReportLab")
bredde, højde = A4

# ---------- Header ----------
# header = [
#    "Lorem ipsum dolor sit amet, consectetur adipiscing elit. Sed do eiusmod tempor incididunt ut labore et dolore magna aliqua.",
#    "Klip sedlerne ud og sæt én på hver ventil."
#]

tekst = (
    "Affotografering af radiatorventiler. Udklip sedlerne og fotografer ventilerne som beskrevet i omsendte mail."
)

header = textwrap.wrap(tekst, width=95)
header.append("Send billederne som vedhæftede filer til abm.ventiler@gmail.com")

# ---------- Layout ----------
margin = 1.5 * cm

header_y = højde - margin
header_h = 2.5 * cm

grid_top = header_y - header_h
grid_h = 24 * cm

cols = 2
rows = 3

left = margin
right = bredde - margin
top = grid_top
bottom = grid_top - grid_h

cell_w = (right - left) / cols
cell_h = (top - bottom) / rows

# ---------- Generér sider ----------
for _, apt in df.iterrows():

    # ---------- Header ----------
    text = c.beginText()
    text.setTextOrigin(left, header_y)
    
    text.setFont("Helvetica", 12)
    text.textLine("Affotografering af radiatorventiler.")
    text.textLine("Udklip sedlerne og fotografer ventilerne som beskrevet i omsendte mail, og send til:")
    
    text.setFont("Courier-Bold", 12)
    text.textLine("abm.ventiler@gmail.com")
    
    c.drawText(text)

    # ---------- Klippelinjer ----------
    c.setDash(3, 3)

    c.line(left + cell_w, bottom, left + cell_w, top)
    c.line(left, top - cell_h, right, top - cell_h)
    c.line(left, top - 2 * cell_h, right, top - 2 * cell_h)

    c.setDash()

    # ---------- Etiketter ----------
    c.setFont("Helvetica-Bold", 16)

    for i, lokale in enumerate(df_ventiler["lokale"]):

        col = i % cols
        row = i // cols

        x = left + col * cell_w
        y = top - (row + 1) * cell_h

        if not apt["har_lokaler"]:
            lokale = ""

        c.drawCentredString(
            x + cell_w / 2,
            y + 1.45 * cm,
            apt["visning"]
        )

        c.drawCentredString(
            x + cell_w / 2,
            y + 0.65 * cm,
            lokale
        )

    c.showPage()

c.save()

print(f"PDF skrevet til {pdf_fil}")

PDF skrevet til data/output/Ventilsedler.pdf
